## FewShotPromptTemplate
> FewShotPromptTemplate -> 입력-출력 예시를 여러 개 보여줘서 모델이 패턴을 학습하게 함

#### 예시

In [ ]:
# 입력과 출력의 패턴을 보여주기 위한 예시 데이터
examples = [
    {"question": "하늘은 왜 파란가요?", "answer": "빛이 산란되기 때문이에요."},
    {"question": "무지개는 왜 생기나요?", "answer": "햇빛이 물방울에서 굴절되기 때문이에요."}
]

In [ ]:
from langchain_core.prompts import PromptTemplate

# 각 예시를 어떤 형식으로 보여줄지 정의
# 예시 + 실제 질문을 합쳐서 모델에 전달할 프롬프트 생성
prompt = PromptTemplate.from_template(
    template="""
    Q: {question}
    A: {answer}
    """
)

In [ ]:
# 프롬프트 안에 들어가야하는 변수
prompt.input_variables

['answer', 'question']

In [8]:
# 지정된 프롬프트 템플릿 출력
print(prompt.template)


    Q: {question}
    A: {answer}
    


### FewShot 프롬프트 생성

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate

# FewShotPromptTemplate: 주어진 예시(few-shot examples)를 포함한 프롬프트 템플릿을 만드는 클래스
fewshot_prompt = FewShotPromptTemplate(
    examples = examples,
    # 모델에게 보여줄 예시 데이터 목록(보통 Q&A나 입력-출력 쌍)
    # 예시를 참고해서 같은 패턴으로 답변하도록 유도
    
    example_prompt = prompt,
    # 각 예시(example)를 어떤 형식으로 출력할지 정의한 프롬프트 템플릿
    # 예: PromptTemplate(input_variables=["question", "answer"], template="Q: {question}\nA: {answer}")

    prefix="다음은 어린이를 위한 과학 질문과 답변예시입니다.\n",
    # 예시들이 출력되기 전에 들어가는 머리말(context)
    # 모델이 어떤 스타일로 답해야 하는지 안내 역할

    suffix="이제 다음 질문에 답해주세요.\nQ: {input}\nA:",
    # 예시들이 출력된 후 실제 입력이 들어가는 부분
    # {input} 자리에 실제 사용자의 질문이 삽입됨
    # 모델이 A: 이후에 답변을 생성하게 됨

    input_variables=["input"]
    # suffix 내에서 사용할 변수 이름을 지정
    # 여기서는 {input}이 실제 질문 텍스트로 대체됨
)

In [12]:
fewshot_prompt.input_variables
# 프롬프트에 들어갈 변수

['input']

In [13]:
print(fewshot_prompt.examples)
# 프롬프트 예시들

[{'question': '하늘은 왜 파란가요?', 'answer': '빛이 산란되기 때문이에요.'}, {'question': '무지개는 왜 생기나요?', 'answer': '햇빛이 물방울에서 굴절되기 때문이에요.'}]


In [14]:
print(fewshot_prompt.example_prompt)
# 예시 프롬프트 형태

input_variables=['answer', 'question'] input_types={} partial_variables={} template='\n    Q: {question}\n    A: {answer}\n    '


In [ ]:
print(fewshot_prompt.format(input="지구는 왜 둥근가요?"))
# 아직 모델을 붙이지 않아서 대답은 없음

다음은 어린이를 위한 과학 질문과 답변예시입니다.



    Q: 하늘은 왜 파란가요?
    A: 빛이 산란되기 때문이에요.
    


    Q: 무지개는 왜 생기나요?
    A: 햇빛이 물방울에서 굴절되기 때문이에요.
    

이제 다음 질문에 답해주세요.
Q: 지구는 왜 둥근가요?
A:


## Ollama Model

In [16]:
from langchain_ollama.chat_models import ChatOllama

model = ChatOllama(
    model="gemma4:e4b",     # 모델 지정
    temperature = 0.1,      # 값이 낮아지면 대답이 정확해지고 안정적이게 됨
    top_p=1.0,              # 전체 분포에서 답을 출력
    num_predict=1000,       # 1000토큰으로 제한해서 출력
    keep_alive="5m"         # 5분동안 모델이 살아있어서 다시 실행했을때 편함
)

## Chain
> 사용자 입력 -> 프롬프트 구성 -> LLM 호출 -> 결과 반환

In [17]:
# Chain 생성
chain = fewshot_prompt | model
# 저장된 chain 정보 불러오기
chain

FewShotPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, examples=[{'question': '하늘은 왜 파란가요?', 'answer': '빛이 산란되기 때문이에요.'}, {'question': '무지개는 왜 생기나요?', 'answer': '햇빛이 물방울에서 굴절되기 때문이에요.'}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Q: {question}\n    A: {answer}\n    '), suffix='이제 다음 질문에 답해주세요.\nQ: {input}\nA:', prefix='다음은 어린이를 위한 과학 질문과 답변예시입니다.\n')
| ChatOllama(model='gemma4:e4b', num_predict=1000, temperature=0.1, top_p=1.0, keep_alive='5m')

In [18]:
# chain 호출
response = chain.invoke({
    "input":"지구는 왜 둥근가요?"
})

In [19]:
# 결과 확인
# 모델에서 받아온 대답에 대해서 content만 출
print(response.content)

지구는 **중력**이라는 힘 때문에 둥근 모양을 유지하고 있어요.

**[추가 설명 (선택 사항):]**
중력은 지구를 이루는 모든 물질을 지구의 중심 쪽으로 계속 당기는 힘이에요. 이 힘이 사방에서 똑같이 작용하기 때문에, 지구는 가장 안정적이고 힘을 적게 받는 '공 모양'으로 뭉치게 된 거랍니다. 마치 풍선을 공기로 가득 채우면 둥근 모양이 되는 것과 비슷해요!
